In [1]:
from unittest import skip
from matplotlib import path
import pandas as pd
from pathlib import Path
import numpy as np
import torch
import h5py

### Load the preprocessed image data, read the h5 file, and convert it into a tensor.

In [2]:
def load_data_and_groundtruth():
    with h5py.File('data//preprocessed//all_uw_data.h5', 'r') as f:
        dataset = torch.tensor(f['dataset'][:])
        gt = torch.tensor(f['groundtruth'][:])
    return dataset, gt

dataset, gt = load_data_and_groundtruth()

### Set the threshold for hypothesis testing.

In [3]:
patientNums = np.arange(100001,100007,1)
SPO2_THRESHOLD = 90.0
gt_mean = gt.mean(dim=1)
labels = (gt_mean < SPO2_THRESHOLD).long()

### Extract 3-second long slices, and for the ground truth data with 1 frame per second, interpolation is required to fill in missing values.

In [4]:
WINDOW = 90 #3 secs,30 frames/sec
STEP = 30 #1 secs

In [5]:
def upsample_gt(gt, fps_ratio=30):
    return gt.repeat_interleave(fps_ratio, dim=1)

gt_upsampled = upsample_gt(gt, fps_ratio=30)
labels = (gt_upsampled < SPO2_THRESHOLD).long()

min_len = min(dataset.shape[2], labels.shape[1])
dataset = dataset[:, :, :min_len]
labels = labels[:, :min_len]

In [6]:
def create_samples(dataset, labels):
    X = []
    Y = []

    for p in range(dataset.shape[0]):
        T = dataset.shape[2]
        for start in range(0, T - WINDOW, STEP):
            end = start + WINDOW

            window = dataset[p, :, start:end]
            label = labels[p, start:end].max()  #峰值，片段中有一帧为低氧则整段label为低氧

            X.append(window)
            Y.append(label)

    return torch.stack(X), torch.tensor(Y)

X, Y = create_samples(dataset, labels)

### Split the data by selecting one participant's data as the validation set, while the data from all other participants serve as the training set.

In [7]:
PATIENT_NUMS = ['100001','100002','100003','100004','100005','100006']
val_patient = '100006' # 选择一名被试作为验证集
val_idx = PATIENT_NUMS.index(val_patient)

In [8]:
def split_val(dataset, labels, val_idx):
    X_train, Y_train = [], []
    X_val,   Y_val   = [], []

    num_patients = dataset.shape[0]
    WINDOW = 90
    STEP = 10

    for p in range(num_patients):
        T = dataset.shape[2]
        for start in range(0, T-WINDOW, STEP):
            end = start + WINDOW
            window = dataset[p, :, start:end]
            label  = labels[p, start:end].max()

            if p == val_idx:
                X_val.append(window)
                Y_val.append(label)
            else:
                X_train.append(window)
                Y_train.append(label)

    return (torch.stack(X_train), torch.tensor(Y_train),
            torch.stack(X_val),   torch.tensor(Y_val))

X_train, Y_train, X_val, Y_val = split_val(dataset, labels, val_idx)

#### Create a Dataset and DataLoader.

In [9]:
from torch.utils.data import Dataset

class PPGDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

from torch.utils.data import DataLoader

train_loader = DataLoader(PPGDataset(X_train, Y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(PPGDataset(X_val,   Y_val),   batch_size=32)

### Use two 1D convolutional layers and one linear layer, and apply dropout to improve generalization and prevent overfitting.

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN_Model(nn.Module):
    def __init__(self):
        super(CNN_Model, self).__init__()

        self.conv1 = nn.Conv1d(in_channels=6, out_channels=32, kernel_size=2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=2)

        self.dropout = nn.Dropout(0.25)

        self.pool = nn.MaxPool1d(kernel_size=2)

        self.fc = nn.Linear(64, 2)

    def forward(self, x):

        x = F.relu(self.conv1(x))
        x = self.pool(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)

        x = self.dropout(x)

        x = torch.mean(x, dim=2)

        x = self.fc(x)

        return x

In [11]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNN_Model().to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### Train for 30 epochs.

In [12]:
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for batch_X, batch_Y in train_loader:
        batch_X = batch_X.to(device)
        batch_Y = batch_Y.to(device)

        pred = model(batch_X)
        loss = loss_fn(pred, batch_Y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # 验证
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for Xv, Yv in val_loader:
            Xv = Xv.to(device)
            Yv = Yv.to(device)
            out = model(Xv)
            pred = out.argmax(1)
            val_correct += (pred == Yv).sum().item()
            val_total   += len(Yv)

    print(f"Epoch {epoch} | loss={train_loss:.3f} | val_acc={val_correct/val_total:.3f}")

Epoch 0 | loss=1.019 | val_acc=1.000
Epoch 1 | loss=0.048 | val_acc=1.000
Epoch 2 | loss=0.003 | val_acc=1.000
Epoch 3 | loss=0.000 | val_acc=1.000
Epoch 4 | loss=0.000 | val_acc=1.000
Epoch 5 | loss=0.000 | val_acc=1.000
Epoch 6 | loss=0.000 | val_acc=1.000
Epoch 7 | loss=0.000 | val_acc=1.000
Epoch 8 | loss=0.000 | val_acc=1.000
Epoch 9 | loss=0.000 | val_acc=1.000
Epoch 10 | loss=0.000 | val_acc=1.000
Epoch 11 | loss=0.000 | val_acc=1.000
Epoch 12 | loss=0.000 | val_acc=1.000
Epoch 13 | loss=0.000 | val_acc=1.000
Epoch 14 | loss=0.000 | val_acc=1.000
Epoch 15 | loss=0.000 | val_acc=1.000
Epoch 16 | loss=0.000 | val_acc=1.000
Epoch 17 | loss=0.000 | val_acc=1.000
Epoch 18 | loss=0.000 | val_acc=1.000
Epoch 19 | loss=0.000 | val_acc=1.000
Epoch 20 | loss=0.000 | val_acc=1.000
Epoch 21 | loss=0.000 | val_acc=1.000
Epoch 22 | loss=0.000 | val_acc=1.000
Epoch 23 | loss=0.000 | val_acc=1.000
Epoch 24 | loss=0.000 | val_acc=1.000
Epoch 25 | loss=0.000 | val_acc=1.000
Epoch 26 | loss=0.000 

An error occurred during the process of adjusting the model. The initial attempt could train and validate the model normally, but since the model converged too quickly, the loss became 0 after just two or three epochs. After trying to increase the number of convolutional layers, the above error occurred, and reverting to previous versions still did not allow it to run properly. The issue may be due to not flattening the dimensions, causing a mismatch in input size.

In [13]:
model.eval()
with torch.no_grad():
    sample = X_val[0:1].to(device)
    out = model(sample)
    prob = out.softmax(dim=1)[0,1].item()

print("P:", prob)

P: 1.0


#### Save and load the model

In [14]:
torch.save(model.state_dict(), "spo2_cnn.pth")

In [15]:
model = CNN_Model()
model.load_state_dict(torch.load("spo2_cnn.pth"))

<All keys matched successfully>